In [ ]:
from descope.utils import load_gene_names_engine
from descope.tokenizer import TokenizerForATAC, tokenize_adata_to_hf_dataset_for_atac

### DeSCOPE

In [ ]:
cellline = ["K562", "K562_1", "K562_2", "MCF7", "GM12878"]  # all cell lines

for c in cellline:
    tokenize_adata_to_hf_dataset_for_atac(
        adata=f"/fse/home/wupengpeng/perturbation_data_origin/ATAC/{c}/{c}.h5ad",
        cell_line_name=c,
        perts_to_exclude=load_gene_names_engine(f"/fse/home/wupengpeng/perturbation_data_origin/ATAC/{c}/Test_Info_{c}.csv"),
        topk_ccres=50000,
        pert_col="perturbation",
        ctrl_name="control",
        save_dir=f"/fse/home/wupengpeng/perturbation_data_origin/ATAC/{c}/tokenized_dataset"
    )

### DeSCOPE_LOO

In [ ]:
cellline = ["K562", "K562_1", "K562_2", "MCF7", "GM12878"]  # all cell lines

for cellline_ft in cellline:
    # cellline_ft: cell line for finetune
    cellline_pt = [c for c in cellline if c != cellline_ft]  # cell line for pretrain

    tokenizer = TokenizerForATAC(
        cell_line_ft=f"/fse/home/wupengpeng/perturbation_data_origin/ATAC/{cellline_ft}/{cellline_ft}.h5ad",
        topk_ccres=50000,
        pert_col="perturbation"
    )

    tokenizer.tokenize(
        # NOTE: Please ensure that the names of control cells in cell_line_pt.obs["pert_col"] are consistent across all datasets.
        cell_line_pt=[
            f"/fse/home/wupengpeng/perturbation_data_origin/ATAC/{cellline}/{cellline}.h5ad" for cellline in cellline_pt
        ],
        cell_line_name=cellline_pt,
        pert_col=["perturbation"] * len(cellline_pt),
        save_dir=f"/fse/home/wupengpeng/perturbation_data_origin/ATAC/tokenized_dataset/loo_{cellline_ft}",
        apply_pert_gene_filter=False
    )